In [11]:
# Cell 1: EM-based UnigramTrainer (50k) with byte-level + byte_fallback

import json
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

DATA_PATH = "/Users/ahmetcanyavuz/Developer/tokenizers/fineweb_data/fineweb_en_sentences.txt"
RAW_JSON_PATH = "final_toks/unigram_em_raw_100k.json"
FINAL_JSON_PATH = "final_toks/unigram_em_bytelevel_100k.json"

SPECIAL_TOKENS = ["<unk>", "<pad>", "<s>", "</s>"]


def line_iterator(path):
    """Lazy line iterator so we don't load the whole file into RAM."""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line:
                yield line


# ----------------- Train EM Unigram -----------------

# 1) Empty Unigram tokenizer + byte-level pre_tokenizer/decoder
tok = Tokenizer(Unigram())
tok.pre_tokenizer = ByteLevel(add_prefix_space=False)
tok.decoder = ByteLevelDecoder()

trainer = UnigramTrainer(
    vocab_size=100_000,          # learned subwords (excluding extra byte tokens we'll add later)
    show_progress=True,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=[],        # ByteLevel handles bytes; no need to preseed alphabet
    shrinking_factor=0.75,
    unk_token="<unk>",
    max_piece_length=16,
    n_sub_iterations=2,
)

tok.train_from_iterator(line_iterator(DATA_PATH), trainer=trainer)

# 2) Save raw JSON (no byte_fallback yet)
tok.save(RAW_JSON_PATH)
print(f"Saved raw EM Unigram to {RAW_JSON_PATH}")


# ----------------- Patch JSON for byte_fallback -----------------

def ensure_unk(model_dict):
    """
    Ensure '<unk>' exists in the Unigram vocab and set unk_id accordingly.
    """
    vocab = model_dict.get("vocab")
    if not isinstance(vocab, list):
        raise ValueError("Unigram JSON missing 'vocab' list")

    unk_idx = None
    for i, (tok, _score) in enumerate(vocab):
        if tok == "<unk>":
            unk_idx = i
            break

    if unk_idx is None:
        vocab.insert(0, ["<unk>", 0.0])
        unk_idx = 0

    model_dict["unk_id"] = unk_idx


def ensure_byte_tokens(model_dict, score=-1.0):
    """
    Inject <0x00>..<0xFF> tokens if missing (with given score, default -1.0).
    """
    vocab = model_dict.get("vocab")
    if not isinstance(vocab, list):
        raise ValueError("Unigram JSON missing 'vocab' list")

    existing = {t for t, _ in vocab}
    added = 0
    for b in range(256):
        t = f"<0x{b:02X}>"
        if t not in existing:
            vocab.append([t, score])
            added += 1
    return added


def patch_unigram_for_byte_fallback(in_path, out_path):
    with open(in_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    model = data.get("model", {})
    if model.get("type") != "Unigram":
        raise ValueError("Model type is not Unigram")

    # 1) Make sure <unk> exists and unk_id is set
    ensure_unk(model)

    # 2) Enable byte_fallback
    model["byte_fallback"] = True

    # 3) Inject full byte token set
    added = ensure_byte_tokens(model, score=-1.0)
    print(f"Injected {added} byte tokens (<0x00>..<0xFF>) into Unigram vocab.")

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    print(f"Saved patched EM Unigram (byte-level) to {out_path}")


patch_unigram_for_byte_fallback(RAW_JSON_PATH, FINAL_JSON_PATH)
unigram_em_bytelevel = Tokenizer.from_file(FINAL_JSON_PATH)

# Quick sanity check
print("Vocab size (including specials + byte tokens):", len(unigram_em_bytelevel.get_vocab()))
print("Example segmentation:", unigram_em_bytelevel.encode("Hello 🙂 world!").tokens)



Saved raw EM Unigram to final_toks/unigram_em_raw_100k.json
Injected 256 byte tokens (<0x00>..<0xFF>) into Unigram vocab.
Saved patched EM Unigram (byte-level) to final_toks/unigram_em_bytelevel_100k.json
Vocab size (including specials + byte tokens): 100256
Example segmentation: ['Hello', 'Ġ', 'ð', 'Ł', 'Ļ', 'Ĥ', 'Ġworld', '!']


In [12]:
# Cell 2: Compression-based Unigram (CompressionTrainer, 50k) with byte-level + byte_fallback

import json
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import CompressionTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

DATA_PATH = "/Users/ahmetcanyavuz/Developer/tokenizers/fineweb_data/fineweb_en_sentences.txt"
RAW_JSON_PATH = "final_toks/unigram_compression_raw_100k.json"
FINAL_JSON_PATH = "final_toks/unigram_compression_bytelevel_100k.json"

SPECIAL_TOKENS = ["<unk>", "<pad>", "<s>", "</s>"]
UNIT_COST = -1.0  # scores used for any extra byte tokens we inject


def line_iterator(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line:
                yield line


# ----------------- Train Compression-based Unigram -----------------

tok = Tokenizer(Unigram())
tok.pre_tokenizer = ByteLevel(add_prefix_space=False)
tok.decoder = ByteLevelDecoder()

trainer = CompressionTrainer(
    vocab_size=100_000,         # learned pieces (not counting extra byte tokens we add later)
    show_progress=True,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=[],       # ByteLevel handles bytes
    max_piece_length=16,
    prune_ratio=0.25,
    seed_size=300_000,
    # seed_vocab=None  # let the trainer build its own seed from data
)
print("trainer initialized")
tok.train_from_iterator(line_iterator(DATA_PATH), trainer=trainer)

tok.save(RAW_JSON_PATH)
print(f"Saved raw compression-based Unigram to {RAW_JSON_PATH}")


# ----------------- Patch JSON for byte_fallback -----------------

def ensure_unk(model_dict):
    vocab = model_dict.get("vocab")
    if not isinstance(vocab, list):
        raise ValueError("Unigram JSON missing 'vocab' list")

    unk_idx = None
    for i, (tok, _score) in enumerate(vocab):
        if tok == "<unk>":
            unk_idx = i
            break

    if unk_idx is None:
        vocab.insert(0, ["<unk>", 0.0])
        unk_idx = 0

    model_dict["unk_id"] = unk_idx


def ensure_byte_tokens(model_dict, score=UNIT_COST):
    vocab = model_dict.get("vocab")
    if not isinstance(vocab, list):
        raise ValueError("Unigram JSON missing 'vocab' list")

    existing = {t for t, _ in vocab}
    added = 0
    for b in range(256):
        t = f"<0x{b:02X}>"
        if t not in existing:
            vocab.append([t, score])
            added += 1
    return added


def patch_unigram_for_byte_fallback(in_path, out_path):
    with open(in_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    model = data.get("model", {})
    if model.get("type") != "Unigram":
        raise ValueError("Model type is not Unigram")

    # special tokens from CompressionTrainer already include "<unk>", but be safe:
    ensure_unk(model)
    model["byte_fallback"] = True

    added = ensure_byte_tokens(model, score=UNIT_COST)
    print(f"Injected {added} byte tokens (<0x00>..<0xFF>) into compression Unigram vocab.")

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    print(f"Saved patched compression Unigram (byte-level) to {out_path}")


patch_unigram_for_byte_fallback(RAW_JSON_PATH, FINAL_JSON_PATH)
unigram_compression_bytelevel = Tokenizer.from_file(FINAL_JSON_PATH)

# Quick sanity check
print("Vocab size (including specials + byte tokens):", len(unigram_compression_bytelevel.get_vocab()))
print("Example segmentation:", unigram_compression_bytelevel.encode("Hello 🙂 world!").tokens)

trainer initialized


[CompressionTrainer] Starting: 1061529 sentences, 300004 initial vocab, 100000 target
[CompressionTrainer] Initial segmentation...
[CompressionTrainer] Computing initial d[t]...
[CompressionTrainer] Pass 1: deleted 50001, vocab_size=250003
[CompressionTrainer] Pass 1 segmentation...
[CompressionTrainer] Pass 1 d[t] (30312 to recompute)...
[CompressionTrainer] Pass 2: deleted 37501, vocab_size=212502
[CompressionTrainer] Pass 2 segmentation...
[CompressionTrainer] Pass 2 d[t] (13603 to recompute)...
[CompressionTrainer] Pass 3: deleted 28126, vocab_size=184376
[CompressionTrainer] Pass 3 segmentation...
[CompressionTrainer] Pass 3 d[t] (11966 to recompute)...
[CompressionTrainer] Pass 4: deleted 21094, vocab_size=163282
[CompressionTrainer] Pass 4 segmentation...
[CompressionTrainer] Pass 4 d[t] (8154 to recompute)...
[CompressionTrainer] Pass 5: deleted 15821, vocab_size=147461
[CompressionTrainer] Pass 5 segmentation...
[CompressionTrainer] Pass 5 d[t] (6360 to recompute)...
[Compress

Saved raw compression-based Unigram to final_toks/unigram_compression_raw_100k.json
Injected 256 byte tokens (<0x00>..<0xFF>) into compression Unigram vocab.
Saved patched compression Unigram (byte-level) to final_toks/unigram_compression_bytelevel_100k.json
Vocab size (including specials + byte tokens): 100256
Example segmentation: ['Hello', 'Ġ', 'ð', 'Ł', 'Ļ', 'Ĥ', 'Ġworld', '!']
